In [1]:
import glob
import os
import pandas as pd
import numpy as np

# Get all match IDs from the offline data folder
files = glob.glob("sb_offline_data/match_*.pkl")
match_ids = [int(os.path.basename(f).split('_')[1].split('.')[0]) for f in files]

print(f"All {len(match_ids)} match IDs found.")

df = pd.read_csv('output/pde/catalogue/events_filtered.csv') # Load PDE catalogue

All 1238 match IDs found.


In [2]:
def player_losing_possession(events, filtered_event_row):
    t = filtered_event_row['total_seconds']
    team = filtered_event_row['affected_team']
    period = filtered_event_row['period']
    
    if 'total_seconds' not in events.columns:
        events['total_seconds'] = events['minute'] * 60 + events['second']
        
    # 15 second window after the event to check for possession loss
    window_events = events[
        (events['total_seconds'] >= t) & 
        (events['total_seconds'] <= t + 15) & 
        (events['period'] == period)
    ].sort_values(['total_seconds', 'timestamp'])
    
    last_player_event = None
    
    for _, ev in window_events.iterrows():
        if ev['team'] == team and pd.notnull(ev.get('player_id')):
            last_player_event = ev # Update the last event for the team in possession
            
        # When the possession changes to the other team, we stop looking further
        if ev.get('possession_team') != team and ev['total_seconds'] > t:
            break
            
    if last_player_event is not None:
        # Calculate the time elapsed since the original event to the last player event
        time_elapsed = last_player_event['total_seconds'] - t
        
        output_row = filtered_event_row.to_dict()
        output_row['total_seconds'] = t + time_elapsed
        
        return [{
            'player_id': last_player_event.get('player_id'),
            'player_name': last_player_event.get('player'),
            'player_position': last_player_event.get('position'),
            **output_row
        }]
    
    return []

In [3]:
# def most_influenced_player(raw_event_row, filtered_event_row, num_players):
#     freeze_frame = raw_event_row.get('shot_freeze_frame')
#     if not isinstance(freeze_frame, list):
#         return []
        
#     shooter_pos = raw_event_row.get('location', [120, 40])
#     affected_team_players = []
    
#     for p in freeze_frame:
#         if not p.get('teammate'): # Only consider players from the affected team
#             player_info = p.get('player', {})
#             position_info = p.get('position', {})
            
#             loc = p.get('location', [0, 0])
#             # Euclidean distance calculation between the shooter and the player
#             dist = np.sqrt((loc[0] - shooter_pos[0])**2 + (loc[1] - shooter_pos[1])**2)
            
#             affected_team_players.append({
#                 'player_id': player_info.get('id'),
#                 'player_name': player_info.get('name'),
#                 'player_position': position_info.get('name'),
#                 'dist': dist
#             })
            
#     # Sort players by distance and select the top N closest players
#     affected_team_players = sorted(affected_team_players, key=lambda x: x['dist'])
#     top_players = affected_team_players[:num_players]
    
#     results = []
#     for tp in top_players:
#         results.append({
#             'player_id': tp['player_id'],
#             'player_name': tp['player_name'],
#             'player_position': tp['player_position'],
#             **filtered_event_row.to_dict()
#         })
        
#     return results

In [4]:
def find_protagonist(events, filtered_event_row):
    raw_event_row = events[events['id'] == filtered_event_row['event_id']].iloc[0]
    event_type = filtered_event_row['event_type']
    if event_type in {'Big Miss', 'Selfish Shot', 'Yellow Card'}:
        return [{
            'player_id': raw_event_row.get('player_id'),
            'player_name': raw_event_row.get('player'),
            'player_position': raw_event_row.get('position'),
            **filtered_event_row.to_dict()
        }]
    if event_type == 'Wasted Turnover':
        return player_losing_possession(events, filtered_event_row)
    if event_type == 'Shock Goal':
        return []

    return []

In [5]:
protagonists = []

# Save the last match_id to avoid reloading the same match data multiple times
last_match_id = None
events = None

for i, filtered_event_row in df.iterrows():
    match_id = filtered_event_row['match_id']
    
    # Only load the match data if the match_id has changed
    if match_id != last_match_id:
        events = pd.read_pickle(f"sb_offline_data/match_{match_id}.pkl")
        last_match_id = match_id

    protagonist_list = find_protagonist(events, filtered_event_row)
    protagonists.extend(protagonist_list)

    if (i + 1) % 500 == 0:
        print(f"Progress: {i + 1}/{len(df)} events completed.")

# Save data to a DataFrame
protagonists_df = pd.DataFrame(protagonists)
if 'player_id' in protagonists_df.columns:
    protagonists_df['player_id'] = protagonists_df['player_id'].astype('Int64')
protagonists_df.to_csv('output/individual/catalogue/protagonists_raw.csv', index=False, encoding='utf-8-sig')


Progress: 500/3449 events completed.
Progress: 1000/3449 events completed.
Progress: 1500/3449 events completed.
Progress: 2000/3449 events completed.
Progress: 2500/3449 events completed.
Progress: 3000/3449 events completed.
